# Facial Emotion Recognition (FER) — GPU 90%+ Framework (Fixed & Extended)
## Benchmark Integration: CK+ • RAVDESS • FER2013 • AffectNet+
### Incorporating DAN, DeepFace Zero-Shot, and Vision Transformer (ViT-B/16)

**Key Improvements & Bug Fixes:**
1. **Fixed Fear & Surprise 0.00 Accuracy:**
   - Fixed RAVDESS globbing to sample all 24 actors uniformly without cutting off at Actor 13.
   - Stratified holdout on CK+ to prevent class depletion.
   - Implemented `WeightedRandomSampler` to enforce equal 1/7 class representation per mini-batch.
   - Implemented **Focal Loss** ($\gamma=2.0$) to focus gradient updates on minority and subtle Action Unit classes.
2. **Advanced Architectural Benchmarking:**
   - **DAN (Distract your Attention Network):** Spatial attention targeting eye and mouth regions.
   - **ViT-B/16 (`vit_base_patch16_224`):** Global patch self-attention.
   - **DeepFace:** Pretrained zero-shot facial attribute analysis.
   - **Quad-Ensemble + TTA:** Combines predictions across CNN, Attention, and Transformer backbones.

### 1. Environment & Hardware Acceleration Setup

In [1]:
import os
import sys
import copy
import json
import math
import time
import random
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import yaml

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

import timm
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

try:
    from deepface import DeepFace
    DEEPFACE_AVAILABLE = True
except ImportError:
    DEEPFACE_AVAILABLE = False
    print("⚠️ DeepFace not installed. Run 'pip install deepface' to enable DeepFace evaluation.")

# Ensure CUDA acceleration
assert torch.cuda.is_available(), "CUDA GPU is required! Ensure PyTorch with CUDA is installed."
device = torch.device("cuda:0")
torch.backends.cudnn.benchmark = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("=" * 75)
print(f"✅ GPU ENFORCED: {torch.cuda.get_device_name(0)}")
print(f"   VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"   CUDA Version  : {torch.version.cuda}")
print("=" * 75)

✅ GPU ENFORCED: NVIDIA GeForce RTX 5090
   VRAM Available: 34.19 GB
   CUDA Version  : 12.8


### 2. Configuration Loading & Directory Initialization

In [2]:
CONFIG_PATH = Path("config_gpu_90plus.yaml")
if not CONFIG_PATH.exists():
    CONFIG_PATH = Path("New Notebook2/config_gpu_90plus.yaml")
if not CONFIG_PATH.exists():
    CONFIG_PATH = Path("../New Notebook2/config_gpu_90plus.yaml")
assert CONFIG_PATH.exists(), f"Configuration file not found: {CONFIG_PATH}"

print(f"Loaded Configuration: {CONFIG_PATH.resolve()}")
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

CANONICAL_CLASSES = CONFIG["label_mapping"]["canonical_classes"]
CLASS2IDX = {c: i for i, c in enumerate(CANONICAL_CLASSES)}
IDX2CLASS = {i: c for c, i in CLASS2IDX.items()}
NUM_CLASSES = len(CANONICAL_CLASSES)

PATHS = CONFIG["paths"]
OUTPUT_DIR     = Path(PATHS.get("output_dir", "./outputs_gpu"))
CHECKPOINT_DIR = Path(PATHS.get("checkpoint_dir", "./checkpoints_gpu"))
CACHE_DIR      = Path(PATHS.get("cache_dir", "./cache_gpu"))
RESULTS_DIR    = Path(PATHS.get("results_dir", "./results_gpu"))

for d in [OUTPUT_DIR, CHECKPOINT_DIR, CACHE_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Canonical Taxonomy:", CLASS2IDX)


Loaded Configuration: C:\Users\Shahzaib Hussain\Desktop\Shahzaib Hussain\FYP\New Notebook2\config_gpu_90plus.yaml
Canonical Taxonomy: {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'sad': 4, 'surprise': 5, 'neutral': 6}


### 3. Multi-Dataset Ingestion (CK+ • RAVDESS • FER2013 • AffectNet+)


In [3]:
def load_ckplus_samples(csv_path):
    df = pd.read_csv(csv_path)
    samples = []
    ck_raw_map = {0: "neutral", 1: "angry", 3: "disgust", 4: "fear", 5: "happy", 6: "sad", 7: "surprise"}
    for idx, row in df.iterrows():
        raw_label = int(row["emotion"])
        c_name = ck_raw_map.get(raw_label)
        if c_name is None:
            continue
        samples.append({
            "source": "ckplus",
            "type": "pixel_str",
            "data": row["pixels"],
            "label": CLASS2IDX[c_name],
            "label_name": c_name,
            "subject_id": f"ck_{idx % 123}"
        })
    print(f"[CK+] Loaded {len(samples)} samples. Distribution: {Counter([s['label_name'] for s in samples])}")
    return samples

def load_ravdess_samples(root_dir, cache_dir):
    manifest_file = cache_dir / "ravdess_balanced_manifest.json"
    rav_frames_dir = cache_dir / "ravdess_frames"
    rav_frames_dir.mkdir(parents=True, exist_ok=True)

    if manifest_file.exists():
        with open(manifest_file, "r", encoding="utf-8") as f:
            samples = json.load(f)
            print(f"[RAVDESS] Loaded {len(samples)} crops across all 24 actors from cached manifest.")
            return samples

    rav_map = {1: "neutral", 3: "happy", 4: "sad", 5: "angry", 6: "fear", 7: "disgust", 8: "surprise"}
    actor_folders = sorted(list(root_dir.glob("Actor_*")))
    if not actor_folders:
        actor_folders = [root_dir]

    cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    samples = []

    for a_folder in actor_folders:
        videos = sorted(list(a_folder.glob("**/*.mp4")))
        for v_p in videos:
            parts = v_p.stem.split("-")
            if len(parts) < 7:
                continue
            try:
                modality = parts[0]
                e_code = int(parts[2])
                actor_id = int(parts[6])
            except ValueError:
                continue

            if modality not in ["01", "02"] or e_code not in rav_map:
                continue

            c_name = rav_map[e_code]
            c_idx = CLASS2IDX[c_name]

            cap = cv2.VideoCapture(str(v_p))
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if total_frames <= 0:
                cap.release()
                continue

            for frac in [0.45, 0.55, 0.65]:
                cap.set(cv2.CAP_PROP_POS_FRAMES, int(total_frames * frac))
                ret, frame = cap.read()
                if not ret:
                    continue
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                faces = cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=4, minSize=(60, 60))
                if len(faces) > 0:
                    x, y, w, h = faces[0]
                    face_crop = cv2.resize(frame[y:y+h, x:x+w], (224, 224))
                else:
                    face_crop = cv2.resize(frame, (224, 224))

                out_p = rav_frames_dir / f"{v_p.stem}_f{int(frac*100)}.png"
                cv2.imwrite(str(out_p), face_crop)

                split = "train" if actor_id <= 18 else ("val" if actor_id <= 21 else "test")
                samples.append({
                    "source": "ravdess",
                    "type": "filepath",
                    "data": str(out_p),
                    "label": c_idx,
                    "label_name": c_name,
                    "split": split,
                    "subject_id": f"actor_{actor_id}"
                })
            cap.release()

    with open(manifest_file, "w", encoding="utf-8") as f:
        json.dump(samples, f)
    print(f"[RAVDESS] Extracted {len(samples)} crops across all 24 actors.")
    return samples

def load_fer2013_samples(root_dir, val_fraction=0.10, seed=42):
    train_dir = root_dir / "train"
    test_dir  = root_dir / "test"
    samples = []

    if train_dir.exists():
        raw_train = []
        for c_name in CANONICAL_CLASSES:
            c_dir = train_dir / c_name
            if not c_dir.exists():
                continue
            c_idx = CLASS2IDX[c_name]
            for img_p in sorted(c_dir.glob("*")):
                if img_p.suffix.lower() in (".jpg", ".jpeg", ".png"):
                    raw_train.append((str(img_p), c_idx, c_name))

        rng = random.Random(seed)
        rng.shuffle(raw_train)
        n_val = int(len(raw_train) * val_fraction)
        val_set = raw_train[:n_val]
        train_set = raw_train[n_val:]

        for p, l, c_name in train_set:
            samples.append({
                "source": "fer2013",
                "type": "filepath",
                "data": p,
                "label": l,
                "label_name": c_name,
                "split": "train",
                "subject_id": f"fer_{Path(p).stem}"
            })
        for p, l, c_name in val_set:
            samples.append({
                "source": "fer2013",
                "type": "filepath",
                "data": p,
                "label": l,
                "label_name": c_name,
                "split": "val",
                "subject_id": f"fer_{Path(p).stem}"
            })

    if test_dir.exists():
        for c_name in CANONICAL_CLASSES:
            c_dir = test_dir / c_name
            if not c_dir.exists():
                continue
            c_idx = CLASS2IDX[c_name]
            for img_p in sorted(c_dir.glob("*")):
                if img_p.suffix.lower() in (".jpg", ".jpeg", ".png"):
                    samples.append({
                        "source": "fer2013",
                        "type": "filepath",
                        "data": str(img_p),
                        "label": c_idx,
                        "label_name": c_name,
                        "split": "test",
                        "subject_id": f"fer_{img_p.stem}"
                    })

    print(f"[FER2013] Loaded {len(samples)} total samples from {root_dir}")
    return samples

def load_affectnet_samples(root_dir, cache_dir, max_per_class=3500, seed=42):
    manifest_file = cache_dir / "affectnet_balanced_manifest.json"
    if manifest_file.exists():
        with open(manifest_file, "r", encoding="utf-8") as f:
            samples = json.load(f)
            print(f"[AffectNet+] Loaded {len(samples)} samples from cached manifest.")
            return samples

    # 0: neutral, 1: happy, 2: sad, 3: surprise, 4: fear, 5: disgust, 6: anger, 7: contempt
    aff_map = {0: "neutral", 1: "happy", 2: "sad", 3: "surprise", 4: "fear", 5: "disgust", 6: "angry"}
    samples = []

    # Validation set
    val_ann_dir = root_dir / "validation_set" / "annotations"
    val_img_dir = root_dir / "validation_set" / "images"
    if val_ann_dir.exists():
        for a_p in sorted(val_ann_dir.glob("*.json")):
            try:
                with open(a_p, "r", encoding="utf-8") as fp:
                    meta = json.load(fp)
                lbl = meta.get("human-label")
                if lbl in aff_map:
                    c_name = aff_map[lbl]
                    img_p = val_img_dir / f"{a_p.stem}.jpg"
                    if img_p.exists():
                        samples.append({
                            "source": "affectnet",
                            "type": "filepath",
                            "data": str(img_p),
                            "label": CLASS2IDX[c_name],
                            "label_name": c_name,
                            "split": "val",
                            "subject_id": f"aff_val_{a_p.stem}"
                        })
            except Exception:
                continue

    # Train set (balanced sampling up to max_per_class)
    tr_ann_dir = root_dir / "train_set" / "annotations"
    tr_img_dir = root_dir / "train_set" / "images"
    if tr_ann_dir.exists():
        class_counts = {c: 0 for c in CANONICAL_CLASSES}
        total_target = max_per_class * len(CANONICAL_CLASSES)
        ann_files = list(tr_ann_dir.glob("*.json"))
        rng = random.Random(seed)
        rng.shuffle(ann_files)

        for a_p in ann_files:
            try:
                with open(a_p, "r", encoding="utf-8") as fp:
                    meta = json.load(fp)
                lbl = meta.get("human-label")
                if lbl in aff_map:
                    c_name = aff_map[lbl]
                    if class_counts[c_name] < max_per_class:
                        img_p = tr_img_dir / f"{a_p.stem}.jpg"
                        if img_p.exists():
                            samples.append({
                                "source": "affectnet",
                                "type": "filepath",
                                "data": str(img_p),
                                "label": CLASS2IDX[c_name],
                                "label_name": c_name,
                                "split": "train",
                                "subject_id": f"aff_tr_{a_p.stem}"
                            })
                            class_counts[c_name] += 1
                            if sum(class_counts.values()) >= total_target:
                                break
            except Exception:
                continue

    with open(manifest_file, "w", encoding="utf-8") as f:
        json.dump(samples, f)
    print(f"[AffectNet+] Extracted {len(samples)} balanced samples (cached to {manifest_file.name}).")
    return samples

# Ingest all 4 datasets
ckplus_csv   = Path(PATHS["ckplus_root"]) / "ckextended.csv"
rav_root     = Path(PATHS["ravdess_root"])
fer2013_root = Path(PATHS["fer2013_root"])
aff_root     = Path(PATHS.get("affectnet_root", "")) / "human_annotated"

print("=" * 70)
print("LOADING ALL 4 BENCHMARK DATASETS:")
print("=" * 70)
ck_samples   = load_ckplus_samples(ckplus_csv) if ckplus_csv.exists() else []
rav_samples  = load_ravdess_samples(rav_root, CACHE_DIR) if rav_root.exists() else []
fer_samples  = load_fer2013_samples(fer2013_root) if fer2013_root.exists() else []
aff_samples  = load_affectnet_samples(aff_root, CACHE_DIR, max_per_class=3500) if aff_root.exists() else []

print("\n" + "=" * 70)
print(f"INGESTION COMPLETE:")
print(f"  • CK+        : {len(ck_samples):>6,} samples")
print(f"  • RAVDESS    : {len(rav_samples):>6,} samples")
print(f"  • FER2013    : {len(fer_samples):>6,} samples")
print(f"  • AffectNet+ : {len(aff_samples):>6,} samples")
print(f"  • TOTAL POOL : {len(ck_samples) + len(rav_samples) + len(fer_samples) + len(aff_samples):>6,} samples")
print("=" * 70)

LOADING ALL 4 BENCHMARK DATASETS:
[CK+] Loaded 895 samples. Distribution: Counter({'sad': 593, 'happy': 83, 'disgust': 69, 'angry': 59, 'neutral': 45, 'fear': 28, 'surprise': 18})
[RAVDESS] Loaded 12456 crops across all 24 actors from cached manifest.
[FER2013] Loaded 35887 total samples from C:\Users\Shahzaib Hussain\Desktop\Shahzaib Hussain\FYP\Datasets\Facial Dataset\FER2013
[AffectNet+] Loaded 28000 samples from cached manifest.

INGESTION COMPLETE:
  • CK+        :    895 samples
  • RAVDESS    : 12,456 samples
  • FER2013    : 35,887 samples
  • AffectNet+ : 28,000 samples
  • TOTAL POOL : 77,238 samples


### 4. Stratified Split Management (Eliminating Zero Class Depletion)

In [4]:
# Stratified 80/10/10 split on CK+
ck_labels = [s["label"] for s in ck_samples]
sss_val_test = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, temp_idx = next(sss_val_test.split(ck_samples, ck_labels))

temp_samples = [ck_samples[i] for i in temp_idx]
temp_labels = [temp_samples[i]["label"] for i in range(len(temp_samples))]
sss_split = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_sub_idx, test_sub_idx = next(sss_split.split(temp_samples, temp_labels))

train_pool = [ck_samples[i] for i in train_idx]
val_pool   = [temp_samples[i] for i in val_sub_idx]
test_pool  = [temp_samples[i] for i in test_sub_idx]

# Append RAVDESS by Actor Holdout
for s in rav_samples:
    if s["split"] == "train":
        train_pool.append(s)
    elif s["split"] == "val":
        val_pool.append(s)
    else:
        test_pool.append(s)

# Append FER2013 (Train, Val, Test)
for s in fer_samples:
    if s["split"] == "train":
        train_pool.append(s)
    elif s["split"] == "val":
        val_pool.append(s)
    else:
        test_pool.append(s)

# Append AffectNet+ (Train, Val)
for s in aff_samples:
    if s["split"] == "train":
        train_pool.append(s)
    else:
        val_pool.append(s)

print("=" * 70)
print(f"UNIFIED 4-DATASET PARTITION COUNTS:")
print(f"  • Train Pool : {len(train_pool):>6,} samples")
print(f"  • Val Pool   : {len(val_pool):>6,} samples")
print(f"  • Test Pool  : {len(test_pool):>6,} samples")
print("=" * 70)

print("\nTest Set Class Distribution:")
test_counts = Counter([IDX2CLASS[s["label"]] for s in test_pool])
for c in CANONICAL_CLASSES:
    print(f"  • {c:<10}: {test_counts.get(c, 0):>5} samples")

UNIFIED 4-DATASET PARTITION COUNTS:
  • Train Pool : 60,343 samples
  • Val Pool   :  8,043 samples
  • Test Pool  :  8,852 samples

Test Set Class Distribution:
  • angry     :  1252 samples
  • disgust   :   262 samples
  • fear      :  1315 samples
  • happy     :  2070 samples
  • sad       :  1594 samples
  • surprise  :   977 samples
  • neutral   :  1382 samples


### 5. Balanced Mini-Batching (WeightedRandomSampler) & Pipeline Transforms

In [5]:
class UnifiedFERDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        if item["type"] == "pixel_str":
            arr = np.fromstring(item["data"], sep=" ", dtype=np.uint8).reshape(48, 48)
            img = Image.fromarray(arr).convert("RGB")
        else:
            img = Image.open(item["data"]).convert("RGB")

        if self.transform:
            img = self.transform(img)
        return img, item["label"]

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=12),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.20, scale=(0.02, 0.08))
])

eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = UnifiedFERDataset(train_pool, transform=train_transforms)
val_ds   = UnifiedFERDataset(val_pool, transform=eval_transforms)
test_ds  = UnifiedFERDataset(test_pool, transform=eval_transforms)

# Balanced sampling so minority classes receive equal mini-batch representation
train_labels = [s["label"] for s in train_pool]
class_counts = Counter(train_labels)
class_weights = {cls: 1.0 / count for cls, count in class_counts.items()}
sample_weights = [class_weights[lbl] for lbl in train_labels]
balanced_sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

BATCH_SIZE = CONFIG["environment"].get("batch_size", 64)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=balanced_sampler, pin_memory=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=0)

print("DataLoaders successfully configured with WeightedRandomSampler.")

DataLoaders successfully configured with WeightedRandomSampler.


### 6. Focal Loss Formulation & Base CNN Architectures

In [6]:
class FocalLoss(nn.Module):
    """Focal Loss: (1 - pt)^gamma * CE prevents majority classes from overpowering gradients."""
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        self.alpha = alpha

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1.0 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

criterion = FocalLoss(gamma=2.0)

class StandardFERModel(nn.Module):
    def __init__(self, timm_name="efficientnet_b0", num_classes=7, pretrained=True, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(timm_name, pretrained=pretrained, num_classes=0)
        in_features = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(p=dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.head(self.backbone(x))

    def get_parameter_groups(self, backbone_lr=1.0e-4, head_lr=3.0e-4, weight_decay=0.05):
        return [
            {"params": self.backbone.parameters(), "lr": backbone_lr, "weight_decay": weight_decay},
            {"params": self.head.parameters(), "lr": head_lr, "weight_decay": weight_decay}
        ]

### 7. Training & Evaluation Engine with Native PyTorch Mixed Precision

In [7]:
def train_one_model(model, model_name, train_loader, val_loader, epochs=None, backbone_lr=1.0e-4, head_lr=3.0e-4):
    if epochs is None:
        epochs = CONFIG.get('training', {}).get('epochs', 30)
    optimizer = optim.AdamW(model.get_parameter_groups(backbone_lr, head_lr), weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
    scaler = torch.amp.GradScaler('cuda')

    best_f1 = 0.0
    best_weights = None
    print(f"\n🚀 Training {model_name} ({epochs} Epochs on {device})...")

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for imgs, targets in train_loader:
            imgs, targets = imgs.to(device, non_blocking=True), targets.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=torch.float16):
                logits = model(imgs)
                loss = criterion(logits, targets)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * imgs.size(0)

        scheduler.step()

        # Validation Loop
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for imgs, targets in val_loader:
                imgs = imgs.to(device)
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    logits = model(imgs)
                val_preds.extend(logits.argmax(dim=1).cpu().numpy())
                val_targets.extend(targets.numpy())

        val_acc = accuracy_score(val_targets, val_preds)
        val_f1  = f1_score(val_targets, val_preds, average='macro', zero_division=0)
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_weights = copy.deepcopy(model.state_dict())
            star = " ★"
        else:
            star = ""

        print(f"Epoch {epoch:02d}/{epochs:02d} | Loss: {running_loss/len(train_loader.dataset):.4f} | "
              f"Val Acc: {val_acc*100:5.2f}% | Val Macro-F1: {val_f1*100:5.2f}%{star}")

    if best_weights:
        model.load_state_dict(best_weights)
    return model

@torch.no_grad()
def predict_probabilities(model, loader, use_tta=True):
    model.eval()
    all_probs, all_targets = [], []
    for imgs, targets in loader:
        imgs = imgs.to(device)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            p1 = F.softmax(model(imgs), dim=1)
            if use_tta:
                p2 = F.softmax(model(torch.flip(imgs, dims=[3])), dim=1)
                probs = 0.5 * (p1 + p2)
            else:
                probs = p1
        all_probs.extend(probs.cpu().numpy())
        all_targets.extend(targets.numpy())
    return np.array(all_probs), np.array(all_targets)

### 8. Train ConvNet Backbones: EfficientNet-B0 & ConvNeXt-Tiny

In [8]:
EPOCHS = CONFIG.get("training", {}).get("epochs", 30)
print(f"Training all backbones for {EPOCHS} Epochs (configured in YAML)...")

model_eff = StandardFERModel("efficientnet_b0", num_classes=NUM_CLASSES).to(device)
model_eff = train_one_model(model_eff, "EfficientNet-B0", train_loader, val_loader, epochs=EPOCHS)

model_cnx = StandardFERModel("convnext_tiny", num_classes=NUM_CLASSES).to(device)
model_cnx = train_one_model(model_cnx, "ConvNeXt-Tiny", train_loader, val_loader, epochs=EPOCHS)

Training all backbones for 30 Epochs (configured in YAML)...



🚀 Training EfficientNet-B0 (30 Epochs on cuda:0)...


KeyboardInterrupt: 

### 9. Pretrained Architecture 1: DAN (Distract your Attention Network)

In [9]:
class DistractAttentionModule(nn.Module):
    """Multi-head spatial attention explicitly weighting emotion-salient action units."""
    def __init__(self, in_channels, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.conv_heads = nn.Conv2d(in_channels, num_heads, kernel_size=1)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        b, c, h, w = x.shape
        attn_maps = self.conv_heads(x)
        attn_weights = self.softmax(attn_maps.view(b, self.num_heads, -1)).view(b, self.num_heads, h, w)
        attended_features = []
        for i in range(self.num_heads):
            w_i = attn_weights[:, i:i+1, :, :]
            feat_i = torch.sum(x * w_i, dim=(2, 3))
            attended_features.append(feat_i)
        return torch.cat(attended_features, dim=-1), attn_weights

class DAN(nn.Module):
    def __init__(self, num_classes=7, num_heads=4):
        super().__init__()
        base = timm.create_model("resnet50", pretrained=True, num_classes=0)
        self.feature_extractor = nn.Sequential(*list(base.children())[:-2])
        self.dam = DistractAttentionModule(in_channels=2048, num_heads=num_heads)
        self.classifier = nn.Sequential(
            nn.Linear(2048 * num_heads, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.feature_extractor(x)
        attended_feats, _ = self.dam(features)
        return self.classifier(attended_feats)

    def get_parameter_groups(self, backbone_lr=1.0e-4, head_lr=3.0e-4, weight_decay=0.05):
        return [
            {"params": self.feature_extractor.parameters(), "lr": backbone_lr, "weight_decay": weight_decay},
            {"params": self.dam.parameters(), "lr": head_lr, "weight_decay": weight_decay},
            {"params": self.classifier.parameters(), "lr": head_lr, "weight_decay": weight_decay}
        ]

print("Initializing DAN (Distract your Attention Network)... ready.")
model_dan = DAN(num_classes=NUM_CLASSES, num_heads=4).to(device)
model_dan = train_one_model(model_dan, "DAN-ResNet50", train_loader, val_loader, epochs=CONFIG.get('training', {}).get('epochs', 30))

Initializing DAN (Distract your Attention Network)... ready.

🚀 Training DAN-ResNet50 (30 Epochs on cuda:0)...


KeyboardInterrupt: 

### 10. Pretrained Architecture 2: Vision Transformer ViT-B/16 (`vit_base_patch16_224`)

In [ ]:
class ViTFERModel(nn.Module):
    def __init__(self, num_classes=7, pretrained=True, dropout=0.3):
        super().__init__()
        self.vit = timm.create_model("vit_base_patch16_224", pretrained=pretrained, num_classes=0)
        in_features = self.vit.num_features
        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(p=dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.head(self.vit(x))

    def get_parameter_groups(self, backbone_lr=5.0e-5, head_lr=3.0e-4, weight_decay=0.05):
        return [
            {"params": self.vit.parameters(), "lr": backbone_lr, "weight_decay": weight_decay},
            {"params": self.head.parameters(), "lr": head_lr, "weight_decay": weight_decay}
        ]

print("Initializing Vision Transformer ViT-B/16... ready.")
model_vit = ViTFERModel(num_classes=NUM_CLASSES, pretrained=True).to(device)
model_vit = train_one_model(model_vit, "ViT-Base-16", train_loader, val_loader, epochs=CONFIG.get('training', {}).get('epochs', 30), backbone_lr=5.0e-5)

Initializing Vision Transformer ViT-B/16... ready.


'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/timm/vit_base_patch16_224.augreg2_in21k_ft_in1k/resolve/main/model.safetensors
Retrying in 1s [Retry 1/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/timm/vit_base_patch16_224.augreg2_in21k_ft_in1k/resolve/main/model.safetensors
Retrying in 2s [Retry 2/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/timm/vit_base_patch16_224.augreg2_in21k_ft_in1k/resolve/main/model.safetensors
Retrying in 4s [Retry 3/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/timm/vit_base_patch16_224.augreg2_in21k_ft_in1k/resolve/main/model.safetensors
Retrying in 8s [Retry 4/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/timm/vit_base_patch16_224.augreg2_in21k_ft_in1k/resolve/main/model.safetensors
Retrying in 8s [Retry 5/5].
'[Errno 11001] getaddrinfo fai

LocalEntryNotFoundError: An error happened while trying to locate the file on the Hub and we cannot find the requested files in the local cache. Please check your connection and try again or make sure your Internet connection is on.

### 11. Pretrained Architecture 3: DeepFace Zero-Shot Evaluation

In [ ]:
deepface_preds = []
deepface_labels = []

if DEEPFACE_AVAILABLE:
    print("\n🔍 Running Pretrained DeepFace Emotion Analyzer on Test Partition...")
    for item in test_pool:
        true_lbl = item["label"]
        try:
            if item["type"] == "pixel_str":
                arr = np.fromstring(item["data"], sep=" ", dtype=np.uint8).reshape(48, 48)
                img_rgb = cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB)
            else:
                img_rgb = cv2.imread(item["data"])
                img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2RGB)

            res = DeepFace.analyze(img_rgb, actions=['emotion'], enforce_detection=False, silent=True)
            pred_emo = res[0]['dominant_emotion'].lower()
            if pred_emo in CLASS2IDX:
                deepface_preds.append(CLASS2IDX[pred_emo])
                deepface_labels.append(true_lbl)
        except Exception:
            continue

    if deepface_preds:
        df_acc = accuracy_score(deepface_labels, deepface_preds)
        df_f1  = f1_score(deepface_labels, deepface_preds, average='macro', zero_division=0)
        print(f"✅ DeepFace Zero-Shot Accuracy: {df_acc*100:.2f}% | Macro-F1: {df_f1*100:.2f}%")
else:
    print("ℹ️ DeepFace module skipped (not installed).")

### 12. Evaluation, Soft-Voting Ensemble & Detailed Per-Class Breakdown

In [ ]:
# Multi-model prediction with TTA
probs_eff, y_test = predict_probabilities(model_eff, test_loader, use_tta=True)
probs_cnx, _      = predict_probabilities(model_cnx, test_loader, use_tta=True)
probs_dan, _      = predict_probabilities(model_dan, test_loader, use_tta=True)
probs_vit, _      = predict_probabilities(model_vit, test_loader, use_tta=True)

# Quad-Ensemble Soft-Voting
ensemble_probs = (probs_eff + probs_cnx + probs_dan + probs_vit) / 4.0
ensemble_preds = np.argmax(ensemble_probs, axis=1)

benchmarks = [
    {"Model": "EfficientNet-B0 + TTA", "Accuracy": f"{accuracy_score(y_test, np.argmax(probs_eff, axis=1))*100:.2f}%", "Macro-F1": f"{f1_score(y_test, np.argmax(probs_eff, axis=1), average='macro'):.4f}"},
    {"Model": "ConvNeXt-Tiny + TTA",   "Accuracy": f"{accuracy_score(y_test, np.argmax(probs_cnx, axis=1))*100:.2f}%", "Macro-F1": f"{f1_score(y_test, np.argmax(probs_cnx, axis=1), average='macro'):.4f}"},
    {"Model": "DAN (Distract Attention)", "Accuracy": f"{accuracy_score(y_test, np.argmax(probs_dan, axis=1))*100:.2f}%", "Macro-F1": f"{f1_score(y_test, np.argmax(probs_dan, axis=1), average='macro'):.4f}"},
    {"Model": "ViT-B/16 (Transformer)",   "Accuracy": f"{accuracy_score(y_test, np.argmax(probs_vit, axis=1))*100:.2f}%", "Macro-F1": f"{f1_score(y_test, np.argmax(probs_vit, axis=1), average='macro'):.4f}"}
]
if DEEPFACE_AVAILABLE and deepface_preds:
    benchmarks.append({"Model": "DeepFace (Zero-Shot)", "Accuracy": f"{df_acc*100:.2f}%", "Macro-F1": f"{df_f1:.4f}"})

benchmarks.append({"Model": "Quad-Ensemble (Eff+Cnx+DAN+ViT)", "Accuracy": f"{accuracy_score(y_test, ensemble_preds)*100:.2f}%", "Macro-F1": f"{f1_score(y_test, ensemble_preds, average='macro'):.4f}"})

df_results = pd.DataFrame(benchmarks)
print("=" * 80)
print("FINAL ACCURACY & F1 BENCHMARK COMPARISON")
print("=" * 80)
print(df_results.to_string(index=False))

print("\nDetailed Per-Class Performance (Verifying Fear & Surprise Recovery):")
print(classification_report(y_test, ensemble_preds, target_names=CANONICAL_CLASSES, digits=4))

### 13. Normalized Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(y_test, ensemble_preds, normalize="true")
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt=".1%", cmap="Blues", xticklabels=CANONICAL_CLASSES, yticklabels=CANONICAL_CLASSES)
plt.title(f"Quad-Ensemble Normalized Confusion Matrix\nAccuracy: {accuracy_score(y_test, ensemble_preds)*100:.2f}%")
plt.ylabel("True Emotion")
plt.xlabel("Predicted Emotion")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ensemble_confusion_matrix.png", dpi=300)
plt.show()